# Context engineering on the original 22-case experiment

## Goal
Show each change between the saved first benchmark and the new model comparison. The original data and final 128 invoice IDs stay fixed. This workflow selects labeled examples; it does not retrain SAP RPT's weights.

## Setup
The first 1,024 rows are the eligible training/context pool. The final 128 invoices, including 22 fraud cases, are the untouched test set. The saved first benchmark used all 1,024 pool rows and its SAP request included both targets. The new comparison uses a 128-row random context and a 128-row class-balanced context, with one target per SAP request. Budget and request schema change between the first benchmark and new results; random versus balanced at 128 rows isolates selection within this dataset.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'archive/synthetic_invoice_experiments/published_baseline/payment_behavior.csv').exists())
sys.path[:0] = [str(ROOT / 'src'), str(ROOT / 'scripts')]
from context_engineering import assert_no_leakage, select_context
from run_context_benchmark import request_for, metric_row
source_path = ROOT / 'archive/synthetic_invoice_experiments/published_baseline/payment_behavior.csv'
source = pd.read_csv(source_path)
pool, holdout = source.iloc[:1024].copy(), source.iloc[1024:].copy()
targets = ('paid_late', 'is_fraud')
seed, budget = 11, 128
assert len(source) == 1152 and len(pool) == 1024 and len(holdout) == 128
assert int(holdout.is_fraud.sum()) == 22 and int(holdout.paid_late.sum()) == 51
assert source.invoice_id.is_unique and not source.isna().any().any()
assert_no_leakage(pool, holdout)


## Step 1: check the data and split
The older eight-case context files are a separate pilot and are not used here. Check IDs, missing values, and target counts before looking at model results.

In [2]:
audit = pd.DataFrame({'slice': ['eligible pool', 'final test'],
                      'rows': [len(pool), len(holdout)],
                      'late_payment_cases': [int(pool.paid_late.sum()), int(holdout.paid_late.sum())],
                      'fraud_cases': [int(pool.is_fraud.sum()), int(holdout.is_fraud.sum())]})
display(audit)
assert set(pool.invoice_id).isdisjoint(holdout.invoice_id)


,slice,rows,late_payment_cases,fraud_cases
0,eligible pool,1024,450,187
1,final test,128,51,22


## Step 2: select the context rows
The **random** condition samples 128 unique labeled rows. The **balanced** condition samples as close to half positive and half negative as the eligible pool permits, separately for each target. Both use the same seed and 128-row budget. This is an explicit selection intervention, not a change to the fraud labels. The earlier eight-case pilot suggested this strategy; its scores are not reused here. This 22-case run is exploratory because strategy choice was informed by that related pilot.

In [3]:
contexts = {}
composition = []
for target in targets:
    for strategy in ('random', 'balanced'):
        selected, _ = select_context(pool, target, holdout.iloc[:1], strategy, budget, seed)
        assert_no_leakage(pool, holdout, selected)
        contexts[(target, strategy)] = selected
        composition.append({'target': target, 'selection': strategy,
                            'rows': len(selected), 'positive_examples': int(selected[target].sum()),
                            'negative_examples': int((selected[target] == 0).sum())})
display(pd.DataFrame(composition))


,target,selection,rows,positive_examples,negative_examples
0,paid_late,random,128,51,77
1,paid_late,balanced,128,64,64
2,is_fraud,random,128,27,101
3,is_fraud,balanced,128,64,64


## Step 3: build a masked SAP prediction request
Only the selected labeled examples supply the target. All final-test target cells are `[PREDICT]`. The other outcome is excluded because it would be unavailable when predicting a new invoice. The invoice ID is an index, not a predictor.

In [4]:
sample = request_for(contexts[('is_fraud', 'balanced')], holdout, 'is_fraud')
assert sample['index_column'] == 'invoice_id'
assert len(sample['rows']) == budget + len(holdout)
assert all(row['is_fraud'] == '[PREDICT]' for row in sample['rows'][budget:])
assert all('paid_late' not in row for row in sample['rows'])
display(pd.DataFrame([{'context_rows': budget, 'prediction_rows': len(holdout),
                       'target': sample['prediction_config']['target_columns'][0]['name'],
                       'query_targets_masked': True, 'other_target_excluded': True}]))


,context_rows,prediction_rows,target,query_targets_masked,other_target_excluded
0,128,128,is_fraud,True,True


## Step 4: train and score the models
XGBoost is fit on the selected rows with the same configuration used by the context benchmark. SAP RPT receives those rows in its request, with weights fixed. `scripts/run_original_context_comparison.py` saves row-level predictions and summary metrics; use `--live` only to fill missing SAP conditions. No call to SAP is made by this notebook.

In [5]:
results_path = ROOT / 'data/original_context_comparison/results.csv'
results = pd.read_csv(results_path)
expected = {(strategy, target, model) for strategy in ('random', 'balanced')
            for target in targets for model in ('XGBoost', 'SAP-RPT')}
available = set(zip(results.strategy, results.target, results.model))
display(results[['strategy', 'target', 'model', 'context_positives', 'true_positives',
                 'false_positives', 'precision', 'recall', 'f1']].round(3))
print('Conditions complete:', len(available & expected), 'of', len(expected))


,strategy,target,model,context_positives,true_positives,false_positives,precision,recall,f1
0,random,paid_late,SAP-RPT,51,23,16,0.590,0.451,0.511
1,random,is_fraud,SAP-RPT,27,10,3,0.769,0.455,0.571
2,balanced,paid_late,SAP-RPT,64,26,26,0.500,0.510,0.505
3,balanced,is_fraud,SAP-RPT,64,16,12,0.571,0.727,0.640
4,random,paid_late,XGBoost,51,27,19,0.587,0.529,0.557
5,random,is_fraud,XGBoost,27,10,20,0.333,0.455,0.385
6,balanced,paid_late,XGBoost,64,32,29,0.525,0.627,0.571
7,balanced,is_fraud,XGBoost,64,16,32,0.333,0.727,0.457


Conditions complete: 8 of 8


## Checks and interpretation
The first benchmark remains the saved 1,024-row run in `notebooks/before_context_engineering/original_128/`. The 128-row random control and balanced treatment use the same source, target labels, test IDs, context selection seed, and model procedures. XGBoost keeps the original model seed (20261002). In this one-seed run, balancing fraud context raises XGBoost detections from 10 to 16 and false alarms from 20 to 32; SAP RPT detections rise from 10 to 16 and false alarms from 3 to 12. The data are synthetic and the test set is fixed, so these changes are descriptive.